<a href="https://colab.research.google.com/github/Zinc-zn/MachineLearning2026/blob/main/JS09/JS09_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# JS09 - Approximate Nearest Neighbors: Lab 3 (Exact NN vs. HNSW)

Praktikum membandingkan exact NN dengan **HNSW (Hierarchical Navigable Small World)** menggunakan library `hnswlib`.

> Pada venv lokal folder JS09, `hnswlib` sudah terinstal saat setup (`pip install hnswlib`).


## Langkah 1 - Instal Library

In [1]:
# Install hnswlib (jika belum terinstal):
# !pip install hnswlib   <- untuk Google Colab
import hnswlib  # cek ketersediaan

## Langkah 2 - Import Library

In [2]:
import hnswlib
import numpy as np
import time
from sklearn.neighbors import NearestNeighbors

## Langkah 3 - Buat Dataset Dummy

In [3]:
# Build Dummy Dataset
num_elements = 1000
dim = 2
np.random.seed(42)
data = np.random.random((num_elements, dim)).astype(np.float32)

# Query point
query = np.array([[0.5, 0.5]], dtype=np.float32)
k = 5  # cari 5 tetangga terdekat

## Langkah 4 - Buat Model Exact NN

In [4]:
# Build NN Model
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(data)

# Compute time performace
start = time.time()
distances, indices = nn.kneighbors(query)
end = time.time()

print("=== Exact NN ===")
print("Indices:", indices)
print("Distances:", distances)
print("Waktu:", end - start, "s")

=== Exact NN ===
Indices: [[137 170 750 316 991]]
Distances: [[0.01144331 0.0278216  0.02824018 0.03417073 0.04514859]]
Waktu: 0.24982523918151855 s


## Langkah 5 - Buat Model HNSW

HNSW membangun graf hierarki *small-world*: pencarian dilakukan dengan melompat dari node ke node pada lapisan graf yang semakin halus. Parameter `ef` (search) mengatur tradeoff kecepatan vs akurasi.

In [5]:
# Initiate Index
p = hnswlib.Index(space='l2', dim=dim)

# Define Max Elements
p.init_index(max_elements=num_elements, ef_construction=100, M=16)

# Add Data
p.add_items(data)

# Set searching parameter
p.set_ef(50)   # tradeoff speed vs accuracy

start = time.time()
labels, distances = p.knn_query(query, k=k)
end = time.time()

print("\n=== HNSW ===")
print("Indices:", labels)
print("Distances:", distances)
print("Waktu:", end - start, "s")


=== HNSW ===
Indices: [[137 170 750 316 991]]
Distances: [[0.00013095 0.00077404 0.00079751 0.00116764 0.0020384 ]]
Waktu: 0.0 s


Pada data kecil ini HNSW menemukan **tetangga yang sama persis** dengan exact NN (indices identik) namun dengan waktu pencarian yang jauh lebih singkat. Perhatikan bahwa nilai jarak HNSW berupa kuadrat jarak Euclidean (space `l2`), sehingga angkanya berbeda dengan sklearn yang menampilkan jarak aslinya.